In [1]:
import sendgrid
import os
import asyncio
import json
from sendgrid.helpers.mail import Mail, Email, To, Content
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
from IPython.display import Markdown, display


# Upload the data

In [2]:
with open("data/hapoel_tel_aviv_season_stats.json") as f:
    hapoel = json.load(f)

with open("data/maccabi_tel_aviv_season_stats.json") as f:
    maccabi = json.load(f)

In [3]:
load_dotenv(override=True)

True

In [4]:
instructions1 = """
You are a Basketball Stats Scout.

Analyze the statistics provided for both teams.

Your responsibilities:
- Identify the best scorers.
- Identify the best rebounders.
- Identify the best playmakers.
- Identify the players with the highest PIR.

Use only the provided statistics.
Do not recommend strategy.
Do not discuss matchups.

Focus only on facts and statistics.
"""


instructions2 = """
You are a Basketball Weakness Scout.

Analyze the statistics provided for both teams.

Your responsibilities:
- Identify players with poor shooting percentages.
- Identify players with high turnover rates.
- Identify players who commit many fouls.
- Identify weaknesses that opponents could exploit.

Use only the provided statistics.
Do not invent information.

Focus on risks, weaknesses, and players that can be targeted.
"""

instructions3 = """
You are a Basketball Matchup Scout.

Analyze the statistics provided for both teams.

Your responsibilities:
- Compare key players from both teams.
- Identify favorable matchups.
- Identify advantages in scoring, rebounding, and playmaking.
- Explain which team has the statistical edge in important positions.

Use only the provided statistics.
Focus on direct player-to-player comparisons.
"""


In [5]:
stats_scout = Agent(
    name="Basketball Stats Scout",
    instructions=instructions1,
    model="gpt-4o-mini"
)

weakness_scout = Agent(
    name="Basketball Weakness Scout",
    instructions=instructions2,
    model="gpt-4o-mini"
)

matchup_scout = Agent(
    name="Basketball Matchup Scout",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [6]:
message = f"""
Analyze both teams.

Hapoel Tel Aviv stats:

{hapoel}

Maccabi Tel Aviv stats:

{maccabi}

Use ONLY the provided statistics.

Do not invent players.
Do not assume missing information.

Provide your analysis.
"""

with trace("Parallel basketball agents"):
    results = await asyncio.gather(
        Runner.run(stats_scout, message),
        Runner.run(weakness_scout, message),
        Runner.run(matchup_scout, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


### Hapoel Tel Aviv Analysis

**Best Scorers:**
1. Elijah Bryant: 15.4 PTS per game
2. Antonio Blakeney: 13.2 PTS per game
3. Dan Oturu: 13.9 PTS per game

**Best Rebounders:**
1. Dan Oturu: 5.7 TR per game
2. Elijah Bryant: 5.2 TR per game
3. Jaylen Hoard: 7.1 TR per game (not applicable to Hapoel)

**Best Playmakers:**
1. Chris Jones: 4.2 AST per game
2. Vasilije Micic: 4.2 AST per game (not applicable to Hapoel)
3. Elijah Bryant: 3.3 AST per game

**Highest PIR:**
1. Elijah Bryant: 19.6 PIR
2. Dan Oturu: 16.8 PIR
3. Chris Jones: 11.8 PIR

---

### Maccabi Tel Aviv Analysis

**Best Scorers:**
1. Jaylen Hoard: 11.7 PTS per game
2. Iffe Lundberg: 11.0 PTS per game
3. Lonnie Walker IV: 15.2 PTS per game

**Best Rebounders:**
1. Jaylen Hoard: 7.1 TR per game
2. O'Shae Brissett: 5.1 TR per game
3. TJ Leaf: 4.7 TR per game

**Best Playmakers:**
1. Tamir Blatt: 5.2 AST per game
2. Jimmy Clark III: 4.3 AST per game
3. Iffe Lundberg: 3.6 AST per game

**Highest PIR:**
1. Jaylen Hoard: 16.1 PI

In [7]:
basketball_analyzer = Agent(
    name="Basketball Analyzer",
    instructions="""
    You are a Basketball Game Analyst.

    You will receive reports from multiple scouting agents.

    Your responsibilities:
    - Review all scout reports.
    - Compare the strengths and weaknesses of both teams.
    - Determine which team is more likely to win.
    - Explain the key reasons for your prediction using the scouts' findings.

    Base your decision only on the information provided by the scouts.
    Do not invent statistics or facts.

    Your final answer must include:
    1. Predicted winner
    2. Confidence level (Low, Medium, or High)
    3. Three main reasons for the prediction
    """,
    model="gpt-4o-mini"
)

In [8]:
message = f"""
Analyze both teams.

Hapoel Tel Aviv stats:

{hapoel}

Maccabi Tel Aviv stats:

{maccabi}

Use ONLY the provided statistics.

Do not invent players.
Do not assume missing information.

Provide your analysis.
"""

with trace("Basketball scouting workflow"):
    results = await asyncio.gather(
        Runner.run(stats_scout, message),
        Runner.run(weakness_scout, message),
        Runner.run(matchup_scout, message),
    )

    scout_outputs = [result.final_output for result in results]

    reports = (
        "Scout Reports:\n\n"
        + "\n\n---\n\n".join(
            [
                f"Report #{i+1}\n\n{output}"
                for i, output in enumerate(scout_outputs)
            ]
        )
    )

    final_result = await Runner.run(
        basketball_analyzer,
        reports
    )

display(Markdown(f"""
# 🏆 Final Analysis

{final_result.final_output}
"""))


# 🏆 Final Analysis

### Predicted Winner
**Hapoel Tel Aviv**

### Confidence Level
**Medium**

### Three Main Reasons for the Prediction**

1. **Scoring Efficiency of Key Players:** Hapoel's leading scorer, Elijah Bryant, shows a remarkable scoring efficiency with a two-point percentage of 59.6% and a three-point percentage of 37.2%. His ability to score consistently gives Hapoel an advantage in terms of offensive output, especially compared to Maccabi's Lonnie Walker IV, whose shooting percentages are lower.

2. **Advantages in Playmaking:** Despite Maccabi having a slight edge in assists with Tamir Blatt, Hapoel's Chris Jones is still a capable playmaker who can facilitate effectively. The combined scoring prowess of Jones and other key players like Dan Oturu may allow Hapoel to exploit Maccabi's defensive weaknesses, particularly in their perimeter defense against poor shooters.

3. **Turnover and Foul Issues:** Both teams have players prone to turnovers and fouls, but Hapoel has identified exploitable weaknesses in its own lineup as well as Maccabi's. Maccabi's Jimmy Clark III has high turnover rates alongside significant foul tendencies which may open up opportunities for Hapoel to capitalize on. This, combined with a strategic focus on pressuring Maccabi's weak shooters, could lead to advantageous possessions for Hapoel.

In conclusion, though the matchup is closely contested and Maccabi has strengths in rebounding and playmaking, Hapoel's higher scoring efficiency and targeted strategies against Maccabi's weaknesses could tip the game in their favor.


## Steps 2 and 3: Tools and Agent interactions


In [9]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body basketball coaches """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("mishkal@mail.yu.edu")  # Change to your verified sender
    to_email = To("aradm1996@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Win/lose email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [10]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body basketball coaches', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x114757ec0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [11]:
description = f"""
Analyze both teams.

Hapoel Tel Aviv stats:

{hapoel}

Maccabi Tel Aviv stats:

{maccabi}

Use ONLY the provided statistics.

Do not invent players.
Do not assume missing information.

Provide your analysis.
"""
tool1 = stats_scout.as_tool(tool_name="stats_scout", tool_description=description)
tool2 = weakness_scout.as_tool(tool_name="weakness_scout", tool_description=description)
tool3 = matchup_scout.as_tool(tool_name="matchup_scout", tool_description=description)

tools = [tool1, tool2, tool3]

# Adding handoffs

In [12]:
email_manager = Agent(
    name="Email Manager",
    instructions="""
    You receive a final basketball analysis.

    Your responsibilities:

    1. Use send_email exactly once.
    2. Send the report exactly as received.
    3. After sending the email, return:

       "Email successfully sent."

    4. Do not perform further analysis.
    5. Do not generate another report.
    6. End the workflow immediately.
    """,
    tools=[send_email],
    model="gpt-4o-mini",
    handoff_description="Send final basketball prediction email"
)

In [14]:
instructions = """
You are a Basketball Game Analyst.

Your goal is to predict the winner using the basketball scout tools.

Steps:

1. Use all basketball scout agents.
2. Review all reports.
3. Determine the most likely winner.
4. Produce a final report containing:

   - Predicted winner
   - Confidence level
   - Three main reasons

5. Hand off the final report to the Email Manager.

Important:
- Use all scout tools.
- Do not send email yourself.
- Hand off exactly one final report.
"""

basketball_game_analyst = Agent(
    name="Basketball Game Analyst",
    instructions=instructions,
    tools=tools,
    handoffs=[email_manager],
    model="gpt-4o-mini"
)
message = f"""
Analyze:

Hapoel:
{hapoel}

Maccabi:
{maccabi}

Predict the winner and email the result.
"""

with trace("Basketball Game Analyst"):
    result = await Runner.run(basketball_game_analyst, message)

In [15]:
print(result)

RunResult:
- Last agent: Agent(name="Email Manager", ...)
- Final output (str):
    Email successfully sent.
- 11 new item(s)
- 5 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
